#### Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import cross_validate
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import root_mean_squared_log_error
import lightgbm as lgb

#### Read train and test data

In [2]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

#### Describe data

In [3]:
print("Train data's size: ", train_data.shape)
print("Test data's size: ", test_data.shape)

Train data's size:  (750000, 9)
Test data's size:  (250000, 8)


In [4]:
numCols = list(train_data.select_dtypes(exclude='object').columns)

# remove "id" from the category columns
numCols.remove("id")
print(f"There are {len(numCols)} numerical features:\n", numCols)

There are 7 numerical features:
 ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']


In [5]:
catCols = list(train_data.select_dtypes(include='object').columns)

print(f"There are {len(catCols)} categorical features:\n", catCols)

There are 1 categorical features:
 ['Sex']


<a name="data-preprocessing"></a>
## Data Preprocessing and Feature Engineering

<a name="feature-add"></a>
## Feature Adder

In [6]:
class FeatureAdder(BaseEstimator, TransformerMixin):
    """
    A custom transformer that adds new features to the dataset.
    Specifically, it extracts the episode number from the 'Episode_Title' column.
    """
    
    def fit(self, X, y=None):
        """
        Fit method for the transformer. Does nothing as this transformer does not require fitting.
        """
        return self
    
    def transform(self, data):
        """
        Transform method to add new features to the dataset.
        """

        data["DIV_HT_WT"] = data["Height"] / data["Weight"]
        data["DIV_BT_DUR"] = data["Body_Temp"] / data["Duration"]
        data["DIV_HR_DUR"] = data["Heart_Rate"] / data["Duration"]
        data["DIV_BT_HR"] = data["Body_Temp"] / data["Heart_Rate"]

        # add log-transform of skewed features
        skewed_feats = ['Body_Temp', 'Height', 'Duration', 'Heart_Rate']
        for feat in skewed_feats:
            data[f'LOG_{feat}'] = np.log1p(data[feat])
        return data


<a name="modeling"></a>
# Modeling

In [7]:
# Define the preprocessing pipeline
preprocessor = Pipeline(
    steps=[
        # Add new features using the custom FeatureAdder transformer
        ("feature_add", FeatureAdder()),
        # Apply column transformations: impute missing values and encode categorical features
        ("column_transform", 
         ColumnTransformer(
            transformers=[
                # Label encode categorical columns, ignoring unknown categories
                ("sex_encoder", OrdinalEncoder(), ["Sex"]),
                ('keep', 'passthrough', ["DIV_HT_WT", "DIV_BT_DUR", "DIV_HR_DUR", "DIV_BT_HR",
                                         "LOG_Body_Temp", "LOG_Height", "LOG_Duration", "LOG_Heart_Rate"]), 
            ])
        )
    ]
)

In [8]:
TARGET = "Calories"

In [9]:
pipeline = Pipeline([("preprocessor", preprocessor),
                     ("lightgbm", lgb.LGBMRegressor(num_threads=-1, learning_rate=0.05, num_leaves=2000, max_depth=-1, n_estimators=2000,
                                                    max_bin=300, objective='regression', random_state=22))])

results = cross_validate(
    pipeline, train_data.drop(TARGET, axis=1), np.log1p(train_data[TARGET]),
    scoring='neg_mean_squared_error',
    cv=5,
    return_train_score=True,
    return_estimator=True,
    verbose=3
)

# Convert to positive MSE
mse_scores = -results["train_score"]
print(f"RMSE scores for each fold, train: {-results['train_score']}, test: {-results['test_score']}")
print("Average RMSE:", np.mean(-results['test_score']))

# Access trained models
fitted_models = results['estimator']

# Make predictions on test data with each model
test_preds = np.column_stack([
    mdl.predict(test_data) for mdl in fitted_models
])

# Average predictions across folds (common strategy)
predictions = np.expm1(test_preds.mean(axis=1))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.034122 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1391
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 9
[LightGBM] [Info] Start training from score 4.141530
[CV] END ................., score=(train=-0.004, test=-0.017) total time= 4.1min
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006812 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1389
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 9
[LightGBM] [Info] Start training from score 4.140934
[CV] END ................., score=(train=-0.004, test=-0.017) total time= 4.2min
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.03

In [10]:
submission_df = pd.DataFrame(test_data["id"])
submission_df[TARGET] = predictions
submission_df.to_csv("data/lightgbm1.csv", index=False)